# Day 3 — Computational Thinking in Action

Computer scientists usually describe **computational thinking** as a handful of overlapping
problem-solving skills that show up again and again, whether you're planning a program on paper
or already deep in the code. Two of the most common ones are **decomposition** and **pattern
recognition** — and today's notebook also builds on a related skill from earlier: **abstraction**.
(There's a fourth pillar, *algorithm design*, that we'll get to later in the course.)

We're going to do this a little backwards on purpose:

- **Part 1** shows you *abstraction* first, using a finished example — so you see the payoff
  before anything else.
- **Part 2** rewinds the tape and shows you the *decomposition* that had to happen to even get
  that example built in the first place.
- **Part 3** shows you *pattern recognition* — how noticing that two pieces have the same shape
  lets you reuse a solution instead of rebuilding it from scratch.

Run each cell in order. Before you run a code cell, try to predict what it will print — then run
it and check yourself.


---
## Part 1 — Abstraction

**Question this pillar answers:** *what can I hide?*

### A tiny online store

Let's say we're building a checkout system. First attempt — we just keep prices in a
dictionary.

In [ ]:
prices = {
    "Laptop": 1200,
    "Monitor": 300,
    "Mouse": 40,
}

print(prices["Laptop"])


That's a completely reasonable way to look something up. Now watch — we're going to replace
that one simple, direct line with a function call that does the exact same thing.

In [ ]:
def get_price(product):
    return prices[product]

print(get_price("Laptop"))


**Before you read on:** we just replaced `prices["Laptop"]` — one easy line — with
`get_price("Laptop")`, a function call that does the *exact same lookup*. Why would anyone ever
do that? Doesn't this just make the code longer for no reason? Jot down your gut answer.

Now hold that thought, because here's why it matters: **we're about to change how prices are
stored, without touching anything that calls `get_price` at all.**


### Attempt 2: prices as a list of pairs

Suppose our data now looks completely different — a list of `[name, price]` pairs instead of a
dictionary.

In [ ]:
products = [
    ["Laptop", 1200],
    ["Monitor", 300],
    ["Mouse", 40],
]

def get_price(product):
    for name, price in products:
        if name == product:
            return price


In [ ]:
# The calling code below is copy-pasted from before. Not one character changed.
print(get_price("Laptop"))


Notice: `print(get_price("Laptop"))` did not change. Only the *inside* of `get_price` changed.

Now compare: if our code had written `prices["Laptop"]` directly, everywhere, switching to a list
would have broken every single one of those lines — and we'd have to hunt down every place that
touched the old dictionary.


### Attempt 3: prices from a real file

Let's push this one step further. A real business wouldn't hardcode prices in the source code —
prices would live in a file (or a database) that changes independently of the program.

Run the cell below once to create that file on disk. Pretend it arrived from someone else's
system — we didn't write it, we don't control its format, we just know it's there.

In [ ]:
with open("prices.csv", "w") as f:
    f.write("name,price\n")
    f.write("Laptop,1200\n")
    f.write("Monitor,300\n")
    f.write("Mouse,40\n")


In [ ]:
import csv

def get_price(product):
    with open("prices.csv") as f:
        reader = csv.reader(f)
        next(reader)  # skip the header row
        for name, price in reader:
            if name == product:
                return int(price)


In [ ]:
# Still the exact same calling code as attempt 1 and attempt 2:
print(get_price("Laptop"))


Three completely different storage formats — a dictionary, a list of pairs, a CSV file on disk —
and the line `get_price("Laptop")` never changed once. **That's the aha moment.**

Whoever calls `get_price` doesn't need to know any of that. They need to know exactly one thing:
*"I can ask for a price."*

```
OLD PROGRAM:  prices["Laptop"]
   depends on knowing:
     • the data is a dictionary
     • "Laptop" is a key
     • the price is the value

ABSTRACT PROGRAM:  get_price("Laptop")
   depends only on knowing:
     • I can ask for a price
```

### The test for a real abstraction

> **Can I change the implementation underneath without changing the code that uses it?**

- If yes → it's an abstraction.
- If changing the implementation forces you to rewrite the caller too → you haven't abstracted
  anything yet. At best, you've decomposed.

> **Decomposition reduces the size of the problem.**
> **Abstraction reduces how much you need to know about the problem.**

Hang on to the first of those two sentences — decomposition, not abstraction, is what Part 2 is
about.


### Scaling it up: a whole checkout

The same idea works for a bigger program. A checkout needs a price, a tax amount, and a shipping
cost for a cart of items. As the *caller*, all you should ever need to write is one line:

In [ ]:
def checkout(cart):
    subtotal = sum(get_price(item) for item in cart)
    tax = get_tax(subtotal)
    shipping = get_shipping(subtotal)
    return subtotal + tax + shipping


`checkout()` doesn't know or care whether `get_price` reads a dictionary, a list, or a CSV file.
It doesn't know *how* `get_tax` computes tax, or *how* `get_shipping` decides on a rate. It just
trusts the **names** of those functions to do their jobs. Let's fill in simple versions so we can
actually run it:

In [ ]:
def get_tax(subtotal):
    return round(subtotal * 0.08, 2)

def get_shipping(subtotal):
    return 0 if subtotal > 75 else 5.99

cart = ["Laptop", "Mouse"]
print(checkout(cart))


Now suppose the company switches tomorrow to a real tax lookup that varies by state. We rewrite
`get_tax` — and `checkout()`, the code everyone else relies on, never has to change:

In [ ]:
state_tax_rates = {"NY": 0.08875, "CA": 0.0725, "OR": 0.0}

def get_tax(subtotal, state="NY"):
    return round(subtotal * state_tax_rates.get(state, 0.0), 2)

print(checkout(cart))  # checkout() was never touched, and it still works


```
messy implementation details
   (dict? list? CSV? a tax API?)
              ↓
         get_price() / get_tax()
              ↑
      simple concept, or "interface"
      "I can ask for a price / a tax amount"
```

**Checkpoint:** `checkout()` showed up above already built, calling three neatly separated
helpers. Fair question — *how did we know to split it into exactly those three pieces in the
first place?* That question is what Part 2 answers.


---
## Part 2 — Decomposition

**Question this pillar answers:** *what are the pieces?*

Let's rewind before `checkout()`, `get_price()`, `get_tax()`, and `get_shipping()` existed at
all. All we started with was one plain-English problem:

> Given a shopping cart, figure out the total the customer owes — item prices, plus tax, plus
> shipping — and print a receipt.

A perfectly natural first attempt is to just... write it. All of it. One function, top to
bottom:

In [ ]:
def process_order(cart):
    # everything mashed into one function
    subtotal = 0
    for item in cart:
        if item == "Laptop":
            subtotal += 1200
        elif item == "Monitor":
            subtotal += 300
        elif item == "Mouse":
            subtotal += 40

    tax = round(subtotal * 0.08, 2)
    shipping = 0 if subtotal > 75 else 5.99
    total = subtotal + tax + shipping

    print(f"Subtotal: ${subtotal}")
    print(f"Tax:      ${tax}")
    print(f"Shipping: ${shipping}")
    print(f"Total:    ${total}")
    return total

process_order(["Laptop", "Mouse"])


This *works*. But sit with it for a second:

- What if another part of the program also needs just the subtotal — say, a "you're $12 away
  from free shipping!" message? You'd have to copy that `for` loop again.
- What if tax rules change? You'd have to go find that logic buried inside a much bigger
  function.
- What if you wanted to test "does tax get calculated correctly?" on its own, without also
  running the shipping logic and the printing logic?

None of those are impossible with `process_order` — they're just *harder than they should be*,
because every sub-task is welded to every other sub-task.

**Decomposition** is the skill of looking at one big problem and noticing the smaller problems
already hiding inside it — then giving each one its own name and its own function.

Looking back at `process_order`, four sub-tasks were tangled together the whole time:

```
process one order
   │
   ├── figure out the subtotal
   ├── figure out the tax
   ├── figure out the shipping
   └── print a receipt
```

Let's pull them apart, one at a time.

In [ ]:
def get_subtotal(cart):
    total = 0
    for item in cart:
        total += get_price(item)
    return total

def get_tax(subtotal):
    return round(subtotal * 0.08, 2)

def get_shipping(subtotal):
    return 0 if subtotal > 75 else 5.99

def print_receipt(subtotal, tax, shipping, total):
    print(f"Subtotal: ${subtotal}")
    print(f"Tax:      ${tax}")
    print(f"Shipping: ${shipping}")
    print(f"Total:    ${total}")

def checkout(cart):
    subtotal = get_subtotal(cart)
    tax = get_tax(subtotal)
    shipping = get_shipping(subtotal)
    total = subtotal + tax + shipping
    print_receipt(subtotal, tax, shipping, total)
    return total

checkout(["Laptop", "Mouse"])


Does that `checkout()` look familiar? It's the exact same function from Part 1 — now you've
seen where it actually comes from. **Decomposition is the step that happens *before*
abstraction.** You can't hide the details of a piece until you've first noticed the piece exists.

They ask different questions:

- Decomposition: *"What are the separate pieces of this problem?"*
- Abstraction: *"Now that I have a piece, what can I hide about how it works?"*

That's also why `get_subtotal`, `get_tax`, and `get_shipping` are each fair game for the same
kind of swap `get_price` went through in Part 1 — pick any one of them, and you could change what
it does internally (a different formula, a different data source, a lookup table instead of
math) without touching `checkout()`. Decomposition is what gave abstraction something to work on
in the first place.

### Decomposition isn't just for code

This is a general thinking skill, not a Python trick. Try it out loud on a problem with zero
code involved: *"Plan a birthday party."* On your own or with a partner, list the smaller
sub-problems hiding inside that one big task (guest list? food? venue? invitations? cleanup?).
Notice that each sub-problem could be handed to a different person to solve independently — that
independence is exactly what makes decomposition useful.


---
## Part 3 — Pattern Recognition

**Question this pillar answers:** *have I seen this shape before?*

Once you've decomposed a problem into pieces, something interesting happens: some of those
pieces turn out to have the *same shape* as pieces you've already solved. Noticing that is
pattern recognition.

Look back at the three versions of `get_price` from Part 1 — the dictionary version, the list
version, and the CSV version. Here's the list version again:

In [ ]:
products = [
    ["Laptop", 1200],
    ["Monitor", 300],
    ["Mouse", 40],
]

def get_price(product):
    for name, price in products:
        if name == product:
            return price


Strip away the words "price" and "product," and what's actually happening?

1. Go through a collection of records, one at a time.
2. Check each record for a match against something you're looking for.
3. Return the value that goes with the match.

That's not a "prices" idea at all — it's a general-purpose recipe, usually called **linear
search**. Let's prove it's a real, reusable pattern (and not a coincidence) by solving a
*completely different* problem the exact same way.

In [ ]:
students = [
    ["Ana", 92],
    ["Ben", 85],
    ["Cy", 78],
]

def get_grade(student):
    for name, grade in students:
        if name == student:
            return grade

print(get_grade("Ben"))


Same shape, different subject. `get_price` looks up a price by product name; `get_grade` looks
up a grade by student name. Once you can see that shape, you stop reinventing it every time and
start reusing it.

That's the real payoff of pattern recognition: instead of writing the same `for` loop over and
over with different variable names, write it **once**, in its most general form:

In [ ]:
def find_match(records, key):
    """records: a list of [key, value] pairs. Returns the value whose key matches, or None."""
    for k, v in records:
        if k == key:
            return v
    return None


In [ ]:
# Now get_price and get_grade are both just find_match wearing a different name:
def get_price(product):
    return find_match(products, product)

def get_grade(student):
    return find_match(students, student)

print(get_price("Monitor"))
print(get_grade("Cy"))


**Wait — wouldn't a dict make more sense here?**

Good instinct, and in real code: yes. `find_match` is reinventing something a Python dict already
does for you — and doing it worse, since checking every record one at a time gets slower as the
list grows, while a dict finds its match almost instantly no matter how big it is. If you were
actually shipping this code, you'd just write `prices[product]` and move on, like Attempt 1 back
in Part 1.

So why write `find_match` at all? Because `prices[product]` on a dict *hides the search* — you
never see it happen, you just get an answer. That's exactly what made it a good abstraction back
in Part 1, but it means there's no loop left to notice here. Writing the list-of-pairs version out
by hand is what let you actually watch the "scan every record, check for a match, return it" shape
happen — which is the whole thing Part 3 is trying to show you.

And here's the callback to Part 1: a Python dict *is* this same pattern, just abstracted away and
made much faster using a technique called hashing, which you'll see later in the course. Every
time you've written `prices["Laptop"]`, Python has been doing a search under the hood — it's just
hidden the mechanism from you so well that it doesn't feel like a search at all.


Notice that the three pillars just worked together — in the order they actually happen when
you build something from scratch, even though we walked through them backwards today:

1. **Decomposition** split "process an order" into smaller pieces like `get_price`.
2. **Pattern recognition** noticed that `get_price`, `get_grade`, and probably a dozen other
   lookups you'll write this semester all share the exact same "search and match" shape.
3. **Abstraction** let you hide that shape behind one reusable name, `find_match`, so every place
   that used to have its own copy of the loop now just asks for a match.

**Try spotting it yourself:** here are four lookups. Which ones are secretly wearing
`find_match`'s shape, and which one isn't?

1. A locker combination lookup, by student ID.
2. A dictionary lookup for a word's definition.
3. A vending machine matching a code to a snack.
4. Finding the highest score in a stack of quiz grades.

Numbers 1–3 are all the exact same shape as `find_match` — a key goes in, its matching value
comes out. Once you can see that shape once, you can build all three the same way. (If your
answer was "they all do," you weren't wrong — you just did pattern recognition correctly on the
examples, which is the whole point.)

Number 4 is the odd one out, and here's why: there's no key to search *for*. You're not asking
"give me the record that matches X" — you're scanning every record and keeping track of the best
one you've seen so far. That's a different, equally common pattern, sometimes called a "running
best":

In [ ]:
def get_highest_grade(grades):
    best = grades[0]
    for grade in grades[1:]:
        if grade > best:
            best = grade
    return best

print(get_highest_grade([92, 85, 78, 99, 61]))


Same tool — a `for` loop — doing a completely different job. Telling "search and match" apart
from "running best" (and, later, other shapes like these two) is exactly the skill Part 3 is
about: it's not just noticing *that* two things loop over data, it's noticing *what kind* of loop
they are.


---
## Putting it together

| Pillar | Question it answers | What we saw today |
|---|---|---|
| Decomposition | What are the pieces? | Splitting `process_order` into subtotal / tax / shipping / receipt |
| Pattern recognition | Have I seen this shape before? | Every `get_price`-style lookup does the same "search and match" dance |
| Abstraction | What can I hide? | `get_price()` hides dict vs. list vs. CSV from everything that calls it |

These three don't happen in a strict order in real life — they loop back on each other constantly.
You decompose a problem, notice a piece looks familiar (pattern recognition), abstract that piece
behind a clean name, and then the *next* problem you decompose reveals more familiar pieces even
faster. That compounding is a big part of what makes experienced programmers fast.

**Before you leave, in your own words:** pick any one function from today — `get_price`,
`checkout`, `find_match`, whatever you like — and name which pillar(s) it demonstrates and why.
